In [1]:
"""
Evaluate syntactic validity of code generated by Ananda100/pocketcoder100M.

This is a CUSTOM nanoGPT-style architecture (not a native `transformers` model),
so it can't be loaded with AutoModel.from_pretrained(). We rebuild the exact
GPTConfig/GPT classes used in training, download config.json + model.safetensors
from the Hugging Face repo, generate completions for a batch of prompts, and
score each generation with Python's own `ast.parse()` to see if it's valid syntax.

Run locally (not inside a network-sandboxed environment) since it needs to reach
huggingface.co:

    pip install torch transformers huggingface_hub safetensors

    export HF_TOKEN="hf_xxx"           # only needed if the repo is private
    python evaluate_syntax_validity.py
"""

import ast
import json
import math
import os
import textwrap
import traceback
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

# --------------------------------------------------------------------------
# Config — edit these two lines
# --------------------------------------------------------------------------
REPO_ID = "Ananda100/100m-sft-python"
HF_TOKEN = os.environ.get("HF_TOKEN")  # rotate any previously-hardcoded token; set via env var instead

TOKENIZER_NAME = "deepseek-ai/deepseek-coder-6.7b-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [2]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer(
                "bias",
                torch.tril(torch.ones(config.block_size, config.block_size)).view(
                    1, 1, config.block_size, config.block_size
                ),
            )

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                drop=nn.Dropout(config.dropout),
                h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                ln_f=LayerNorm(config.n_embd, config.bias),
            )
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, (
            f"sequence length {t} exceeds block_size {self.config.block_size}"
        )
        pos = torch.arange(0, t, dtype=torch.long, device=dev)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx


# --------------------------------------------------------------------------
# Load model + tokenizer from the Hub
# --------------------------------------------------------------------------

def load_model_from_hub(repo_id: str, token: str | None):
    print(f"Downloading config.json and model.safetensors from {repo_id} ...")
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json", token=token)
    weights_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors", token=token)

    with open(config_path) as f:
        cfg_dict = json.load(f)
    config = GPTConfig(**cfg_dict)
    print("Loaded config:", asdict(config))

    model = GPT(config)
    state_dict = load_file(weights_path)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print("WARNING - missing keys:", missing)
    if unexpected:
        print("WARNING - unexpected keys:", unexpected)

    model.to(DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model loaded: {n_params:.2f}M params on {DEVICE}")
    return model, config


def load_tokenizer(repo_id: str, token: str | None):
    # Try the tokenizer files saved in the repo itself first (matches training exactly);
    # fall back to the original DeepSeek Coder tokenizer if the repo doesn't have them.
    try:
        tok = AutoTokenizer.from_pretrained(repo_id, token=token, trust_remote_code=True)
        print(f"Loaded tokenizer from {repo_id}")
    except Exception:
        print(f"No tokenizer in {repo_id}, falling back to {TOKENIZER_NAME}")
        tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)
    return tok


# --------------------------------------------------------------------------
# Generation
# --------------------------------------------------------------------------

def generate_code(model, tokenizer, prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        out = model.generate(
            idx,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0].tolist(), skip_special_tokens=True)


PROMPTS = [
    # --- imports / async / stdlib-heavy (30) ---
    'import asyncio\n\nasync def',
    'import pandas as pd\n\ndef load_data(path):',
    'import numpy as np\n\ndef normalize(vector):',
    'import json\n\ndef parse_config(filepath):',
    'import os\n\ndef list_python_files(directory):',
    'import re\n\ndef extract_emails(text):',
    'import requests\n\ndef fetch_json(url):',
    'from typing import List, Optional\n\ndef find_max(values: List[int]) -> Optional[int]:',
    'import argparse\n\ndef parse_args():',
    'import logging\n\nlogger = logging.getLogger(__name__)\n\ndef setup_logging():',
    'import socket\n\ndef create_server(host, port):',
    'import subprocess\n\ndef run_command(cmd):',
    'import threading\n\ndef start_worker():',
    'import multiprocessing\n\ndef parallel_map(func, items):',
    'import sqlite3\n\ndef connect_db(path):',
    'import csv\n\ndef read_csv_file(path):',
    'import itertools\n\ndef pairwise(iterable):',
    'import collections\n\ndef word_frequency(text):',
    'import datetime\n\ndef days_until(target_date):',
    'from pathlib import Path\n\ndef find_files(directory, extension):',
    'import hashlib\n\ndef compute_sha256(filepath):',
    'import random\n\ndef shuffle_deck():',
    'import unittest\n\nclass TestMath(unittest.TestCase):',
    'import pytest\n\ndef test_addition():',
    "from flask import Flask\n\napp = Flask(__name__)\n\n@app.route('/health')\ndef",
    'import boto3\n\ndef upload_to_s3(bucket, key, filepath):',
    'import torch\n\ndef create_tensor(shape):',
    'from sklearn.model_selection import train_test_split\n\ndef split_dataset(X, y):',
    'import xml.etree.ElementTree as ET\n\ndef parse_xml(filepath):',
    'import urllib.request\n\ndef download_file(url, dest):',

    # --- classes / OOP (30) ---
    'class BankAccount:\n    def __init__(self, balance):',
    'class Node:\n    def __init__(self, value):',
    'class Stack:',
    'class Queue:\n    def __init__(self):\n        self.items = []',
    'class Animal:\n    def __init__(self, name):\n        self.name = name\n\nclass Dog(Animal):',
    'class Shape(ABC):\n    @abstractmethod\n    def area(self):',
    '@dataclass\nclass Point:',
    'class Singleton:\n    _instance = None\n\n    def __new__(cls):',
    'class LinkedList:\n    def __init__(self):\n        self.head = None\n\n    def append(self, value):',
    'class BinarySearchTree:\n    def __init__(self):\n        self.root = None\n\n    def insert(self, value):',
    'class Graph:\n    def __init__(self):\n        self.adjacency = {}\n\n    def add_edge(self, u, v):',
    'class Trie:\n    def __init__(self):\n        self.children = {}\n\n    def insert(self, word):',
    'class MinHeap:\n    def __init__(self):\n        self.data = []\n\n    def push(self, value):',
    'class Observer(ABC):\n    @abstractmethod\n    def update(self, event):',
    'class Factory:\n    @staticmethod\n    def create(kind):',
    'class Vector:\n    def __init__(self, x, y):\n        self.x = x\n        self.y = y\n\n    def __add__(self, other):',
    'class Matrix:\n    def __init__(self, rows, cols):',
    'class Employee:\n    def __init__(self, name, salary):\n        self.name = name\n        self.salary = salary\n\n    def __repr__(self):',
    'class Cache:\n    def __init__(self, capacity):\n        self.capacity = capacity\n        self.data = {}\n\n    def get(self, key):',
    'class EventEmitter:\n    def __init__(self):\n        self.listeners = {}\n\n    def on(self, event, callback):',
    'class Rectangle:\n    def __init__(self, width, height):\n        self.width = width\n        self.height = height\n\n    def __eq__(self, other):',
    'class Iterator:\n    def __init__(self, data):\n        self.data = data\n        self.index = 0\n\n    def __next__(self):',
    'class ContextTimer:\n    def __enter__(self):',
    'class Config:\n    def __init__(self, **kwargs):',
    'class StateMachine:\n    def __init__(self, initial_state):\n        self.state = initial_state\n\n    def transition(self, event):',
    'class PriorityQueue:\n    def __init__(self):\n        self.heap = []\n\n    def push(self, item, priority):',
    "class Point3D:\n    __slots__ = ('x', 'y', 'z')\n\n    def __init__(self, x, y, z):",
    'class LRUCache:\n    def __init__(self, capacity: int):',
    'class TreeNode:\n    def __init__(self, val=0, left=None, right=None):',
    'class Polynomial:\n    def __init__(self, coefficients):\n        self.coefficients = coefficients\n\n    def evaluate(self, x):',

    # --- algorithms: sorting / searching (25) ---
    'def bubble_sort(arr):',
    'def quicksort(arr):',
    'def merge_sort(arr):',
    'def insertion_sort(arr):',
    'def selection_sort(arr):',
    'def heap_sort(arr):',
    'def binary_search(arr, target):',
    'def linear_search(arr, target):',
    'def breadth_first_search(graph, start):',
    'def depth_first_search(graph, start, visited=None):',
    'def dijkstra(graph, start):',
    'def topological_sort(graph):',
    'def kth_smallest(arr, k):',
    'def find_median(arr):',
    'def counting_sort(arr):',
    'def radix_sort(arr):',
    'def binary_search_insert_position(arr, target):',
    'def two_sum(nums, target):',
    'def three_sum(nums):',
    'def rotate_array(arr, k):',
    'def find_duplicates(arr):',
    'def merge_intervals(intervals):',
    'def sliding_window_max(nums, k):',
    'def quickselect(arr, k):',
    'def bucket_sort(arr):',

    # --- algorithms: strings / math (25) ---
    'def fibonacci(n):',
    'def is_prime(n):',
    'def gcd(a, b):',
    'def lcm(a, b):',
    'def factorial(n):',
    'def is_palindrome(s):',
    'def reverse_string(s):',
    'def count_words(text):',
    'def sieve_of_eratosthenes(n):',
    'def is_anagram(s1, s2):',
    'def longest_common_subsequence(s1, s2):',
    'def edit_distance(s1, s2):',
    'def knapsack(weights, values, capacity):',
    'def power_set(items):',
    'def permutations_of(items):',
    'def combinations_of(items, r):',
    'def caesar_cipher(text, shift):',
    'def run_length_encode(text):',
    'def is_balanced_parentheses(s):',
    'def string_to_int(s):',
    'def roman_to_int(s):',
    'def int_to_roman(num):',
    'def longest_palindromic_substring(s):',
    'def word_break(s, word_dict):',
    'def matrix_multiply(a, b):',

    # --- data structures utility (15) ---
    'def flatten(nested_list):',
    'def merge_dicts(a, b):',
    'def deep_copy(obj):',
    'def group_by(items, key_func):',
    'def chunk_list(lst, size):',
    'def remove_duplicates(lst):',
    'def transpose_matrix(matrix):',
    'def zip_dicts(a, b):',
    'def invert_dict(d):',
    "def flatten_dict(d, parent_key=''):",
    'def most_common_element(lst):',
    'def unique_by_key(items, key):',
    'def partition(lst, predicate):',
    'def moving_average(values, window):',
    'def dict_to_namedtuple(d):',

    # --- control flow / context managers (20) ---
    "with open('data.txt') as f:",
    'try:',
    'try:\n    result = risky_operation()\nexcept',
    'for i in range(10):',
    'while True:',
    "if __name__ == '__main__':",
    "with open('input.csv') as f, open('output.csv', 'w') as g:",
    'try:\n    value = int(user_input)\nexcept ValueError:',
    'with contextlib.suppress(FileNotFoundError):',
    'for key, value in sorted(data.items()):',
    'while queue:',
    'for index, item in enumerate(items):',
    'with tempfile.TemporaryDirectory() as tmpdir:',
    'try:\n    conn = connect_to_db()\nfinally:',
    'for chunk in read_in_chunks(file, chunk_size=1024):',
    'with lock:',
    'for row in cursor.fetchall():',
    'while not queue.empty():',
    'with suppress_warnings():',
    'for batch in dataloader:',

    # --- comprehensions / decorators / generators (15) ---
    'squares = [x**2 for x in range(10) if',
    'def cache(func):\n    @wraps(func)\n    def wrapper(*args, **kwargs):',
    'def read_large_file(path):\n    with open(path) as f:\n        for line in f:\n            yield',
    'def timer(func):',
    'def retry(max_attempts=3):\n    def decorator(func):',
    'even_squares = {x: x**2 for x in range(20) if x % 2 == 0',
    'def fibonacci_generator():',
    'def batch_generator(items, batch_size):',
    'def log_calls(func):\n    def wrapper(*args, **kwargs):',
    'unique_items = {item for item in',
    'def validate_args(func):',
    'def infinite_counter(start=0):',
    'def memoize(func):\n    cache = {}\n    def wrapper(*args):',
    'def rate_limited(calls_per_second):',
    'def lazy_property(func):',

    # --- typing / data pipelines / ML (20) ---
    'def parse_csv_row(row: str) -> dict:',
    'async def fetch_all(urls: list[str]) -> list[dict]:',
    'def train_model(X_train, y_train, epochs=10):',
    'def load_data(path):\n    df = pd.read_csv(path)\n    return',
    'def preprocess_text(text: str) -> list[str]:',
    'def compute_accuracy(y_true: list[int], y_pred: list[int]) -> float:',
    'def batch_iterator(dataset, batch_size: int):',
    'class Dataset:\n    def __getitem__(self, idx: int):',
    'def normalize_image(image: np.ndarray) -> np.ndarray:',
    'def encode_labels(labels: list[str]) -> dict[str, int]:',
    'def cross_validate(model, X, y, folds: int = 5):',
    'def save_checkpoint(model, path: str) -> None:',
    'def load_checkpoint(path: str):',
    'def compute_loss(predictions, targets):',
    'def tokenize(text: str) -> list[str]:',
    'def build_vocab(sentences: list[list[str]]) -> dict:',
    'def pad_sequences(sequences: list[list[int]], max_len: int):',
    'def one_hot_encode(labels: list[int], num_classes: int):',
    'def evaluate_model(model, dataloader):',
    'def early_stopping(val_losses: list[float], patience: int = 5) -> bool:',

    # --- error handling / validation (10) ---
    'def validate_email(email):',
    'def safe_divide(a, b):',
    'class ValidationError(Exception):',
    'def assert_positive(value):',
    'def retry_on_failure(func, max_retries=3):',
    'def check_file_exists(filepath):',
    'class CustomTimeoutError(Exception):\n    def __init__(self, message, timeout):',
    'def validate_json_schema(data, schema):',
    'def raise_if_negative(value):',
    'def handle_api_error(response):',

    # --- deliberately awkward / stress-test prompts (10) ---
    'def unclosed_function(',
    "x = {'a': 1, 'b': [1, 2, 3",
    'def f():\n    return (1 +',
    's = "this string never',
    'class Broken(',
    'for i in range(',
    'def deeply_nested():\n    if True:\n        if True:\n            if True:\n                return (((1 + 2)',
    'lambda x, y:',
    'result = [i for i in range(10) if i %',
    'def multi_return(a, b):\n    if a > b:\n        return a\n    elif',

]


# --------------------------------------------------------------------------
# Syntax validity scoring
# --------------------------------------------------------------------------

def classify_syntax_error(code: str, error: SyntaxError) -> str:
    msg = str(error).lower()
    if "unterminated string" in msg or "eol while scanning" in msg:
        return "unterminated_string"
    if "unexpected eof" in msg or "unexpected end of file" in msg:
        return "unexpected_eof_truncation"
    if "indent" in msg:
        return "indentation_error"
    if "unmatched" in msg or "was never closed" in msg or "closing parenthesis" in msg:
        return "unbalanced_brackets"
    if "invalid syntax" in msg:
        return "invalid_syntax_other"
    return "other"


def check_syntax(code: str):
    """Returns (is_valid, error_category or None, error_message or None)."""
    try:
        ast.parse(code)
        return True, None, None
    except SyntaxError as e:
        return False, classify_syntax_error(code, e), str(e)
    except Exception as e:  # tokenize errors etc.
        return False, "other", str(e)


def evaluate(model, tokenizer, prompts=PROMPTS, max_new_tokens=200, temperature=0.8, top_k=50,
             samples_per_prompt=1, verbose=True):
    """
    samples_per_prompt > 1 lets you generate multiple completions per prompt
    (useful since sampling is stochastic) and report validity rate across all samples.
    """
    results = []
    total = len(prompts) * samples_per_prompt
    done = 0
    for prompt in prompts:
        for _ in range(samples_per_prompt):
            done += 1
            completion = generate_code(model, tokenizer, prompt, max_new_tokens, temperature, top_k)
            is_valid, category, err_msg = check_syntax(completion)
            results.append(
                {
                    "prompt": prompt,
                    "completion": completion,
                    "is_valid": is_valid,
                    "error_category": category,
                    "error_message": err_msg,
                }
            )

            if verbose:
                print("=" * 70)
                print(f"[{done}/{total}] PROMPT:", repr(prompt))
                print("-" * 70)
                print(completion)
                print("-" * 70)
                status = "VALID" if is_valid else f"INVALID ({category}: {err_msg})"
                print("SYNTAX:", status)

    n_valid = sum(r["is_valid"] for r in results)
    n_total = len(results)
    rate = n_valid / n_total if n_total else 0.0

    print("\n" + "=" * 70)
    print(f"OVERALL SYNTAX VALIDITY: {n_valid}/{n_total} = {rate:.1%}")
    print(f"(across {len(prompts)} distinct prompts x {samples_per_prompt} sample(s) each)")

    error_counts = {}
    for r in results:
        if not r["is_valid"]:
            error_counts[r["error_category"]] = error_counts.get(r["error_category"], 0) + 1
    if error_counts:
        print("\nError breakdown:")
        for cat, count in sorted(error_counts.items(), key=lambda x: -x[1]):
            print(f"  {cat}: {count} ({count / n_total:.1%} of all generations)")

    if samples_per_prompt > 1:
        print("\nPer-prompt validity:")
        for prompt in prompts:
            prompt_results = [r for r in results if r["prompt"] == prompt]
            v = sum(r["is_valid"] for r in prompt_results)
            print(f"  {v}/{len(prompt_results)}  {prompt[:50]!r}")

    return results, rate


if __name__ == "__main__":
    model, config = load_model_from_hub(REPO_ID, HF_TOKEN)
    tokenizer = load_tokenizer(REPO_ID, HF_TOKEN)

    # Bump samples_per_prompt to e.g. 3 to average out sampling noise per prompt
    results, rate = evaluate(model, tokenizer, samples_per_prompt=1)

    with open("syntax_validity_results.json", "w") as f:
        json.dump({"validity_rate": rate, "results": results}, f, indent=2)
    print("\nSaved detailed results to syntax_validity_results.json")

config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  482MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded config: {'block_size': 512, 'vocab_size': 32022, 'n_layer': 10, 'n_head': 12, 'n_embd': 768, 'dropout': 0.1, 'bias': True}
Model loaded: 95.87M params on cuda


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.29M [00:00<?, ?B/s]

Loaded tokenizer from Ananda100/100m-sftd-python
[1/49] PROMPT: 'import asyncio\n\nasync def'
----------------------------------------------------------------------
import asyncio

async def async_process_events(events, event_id):
    for event in events:
        await asyncio.sleep(event.data[event_id])

# Example usage:
async def test_event_with_timeout(loop):
    with asyncio.ensure_future(loop) as s:
        async with event_with_timeout(5, timeout=3) as msg:
            await msg.set_timeout(3)
    assert await asyncio.wait_for(lambda: len(s.get_events()) == 3)

async def test_event_with_timeout_concurrently(loop):
    with asyncio.ensure_future(loop) as s:
        async with event_with_timeout(5, timeout=3) as msg:
            await s.set_timeout(3)
    assert await asyncio.wait_for(lambda: len
----------------------------------------------------------------------
SYNTAX: INVALID (unbalanced_brackets: '(' was never closed (<unknown>, line 18))
[2/49] PROMPT: 'import pandas as pd\